## This notebook will test streamfit on one parameter at a time, keeping all others at the best fit parameters

In [ ]:
# Imports
import cProfile
import pstats
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad, outputs
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
iras2a_ref = iras2a_c.skyoffset_frame()
distance_iras2a = 293 #parsecs
# choose which distance
distance = distance_iras2a

# cubefile = 'test_data/HLTau/HLTAU_HCOp32.fits'a
# file_Tpeak = 'test_data/HLTau/HLTAU_HCOp32_Tpeak.fits'
cubefile = '../test_data/IRAS2A/D2CO_streamer_cluster_data.fits'
file_Tpeak = '../test_data/IRAS2A/D2CO_streamer_cluster_tpeak.fits'

# some constants
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km



### Original cube with spectra: Prepare the 1D streamer emission from the cube

In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)

# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer

'''
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer
'''

vmin = 6
vmax = 8
xmin = -5
xmax = 5
ymin = -12
ymax = 0.5
rms_thresh = 4

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 


In [ ]:
n_points = 10 # the number of points we want to reduce the data to


# Extract 1D streamline from the data cube
pc_coords, pc_means, pc_stds = extract_streamline.reduce_to_1D(streamer_cube, yso_centre=iras2a_c,n_elements=n_points)
print(f"point cloud velocities (km/s): {pc_coords[2]}")

# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)
print(f"data velocities (km/s): {v_data}")

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]

rproj = np.sqrt(ra_data**2 + dec_data**2)
print(f"projected distances from star (arcsec): {rproj}")

data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

In [ ]:
# plot the extracted morphology
outputs.plot_morphology(
    ra_data=ra_data,
    dec_data=dec_data,
    ra_sigma=ra_sigma,
    dec_sigma=dec_sigma,
    pc_coords=pc_coords,
    show=True,
)


## Set the parameters of the best-fit

In [ ]:
# Parameters to optimize
best_fit_params = {
    'r0': 2702,  # au
    'theta0': 9.052e-01 * 180 / np.pi,  # degrees
    'phi0': 1.086e+00 * 180 / np.pi,  # degrees
    'log_omega': -2.823e+01,  # log(1/s)
    'v_r0': 0.001,  # km/s
}

# IRAS2A
fixed_params = {
    'mass': 4.0,  # solar masses
    'inc': -45.0,  # degrees
    'pa': 194.0,  # degrees
    'rmin': 50.0,  # au
    'deltar':50.0,  # au
    'v_lsr': 7.5  # km/s (systemic velocity)
}

# Convert angles from degrees to radians
best_fit_params['theta0'] = np.radians(best_fit_params['theta0'])
best_fit_params['phi0'] = np.radians(best_fit_params['phi0'])
fixed_params['inc'] = np.radians(fixed_params['inc'])
fixed_params['pa'] = np.radians(fixed_params['pa'])

def get_omega(mass, r0):
    '''
    this gets value of omega when r_cent = 0.5 * r0
    '''
    omega_squared = 0.5 * G * mass / (jnp.power(r0, 3) * jnp.power(au_in_km, 2)) # in s^-2
    omega = jnp.power(omega_squared, 0.5) # in s^-1
    return omega

best_params = best_fit_params.copy()

# Define physically reasonable bounds (omega bounds transformed to natural log space)
# These bounds are also used as normalization anchors: x_norm = (x - min) / (max - min).
# Provide bounds for every optimized parameter.
r0_min, r0_max = 200.0, 10000.0 # param bounds in au

# the omega bounds are set by keeping centrifugal radius reasonable (r_cent = 0.5 r0)
omega_max = get_omega(fixed_params['mass'], r0_min)
omega_min = get_omega(fixed_params['mass'], r0_max)
# print these in scientific notation for sanity check
print(f"Omega bounds: {omega_min:.2e} to {omega_max:.2e} 1/s")

param_bounds = {
    'r0': (r0_min, r0_max),                    # radius between 200-10000 au
    'theta0': (0.0, np.pi),                    # polar angle 0-pi
    'phi0': (0.0, 2*np.pi),                    # azimuthal angle 0-2pi
    'log_omega': (np.log(omega_min), np.log(omega_max)),  # omega in [omega_min, omega_max] 1/s
    'v_r0': (-10.0, 10.0),                       # radial velocity 0.01 to 2 km/s
}

## Parameter sweep function

In [ ]:
# Define helper function for parameter sweeps
def run_param_sweep(param, n_initials=8, n_epochs=100, learning_rate=0.01, loss_method='rthetavel',
                     gradient_tol=1e-2, sweep_output_root="streamfit_test_output/param_sweeps"):
    """
    Run a single-parameter sweep: vary one parameter across its bounds and optimize.
    
    Parameters
    ----------
    param : str
        Name of the parameter to sweep (must be in best_fit_params and param_bounds)
    n_initials : int
        Number of initial guesses to sample across the parameter bounds
    n_epochs : int
        Number of epochs for each optimization run
    learning_rate : float
        Learning rate for gradient descent
    loss_method : str
        Loss method ('radecvel' or 'rthetavel')
    gradient_tol : float
        Gradient tolerance for early stopping
    sweep_output_root : str
        Root directory for output CSVs and plots
    
    Returns
    -------
    df_all : pandas.DataFrame
        Aggregated results from all runs for this parameter
    """
    if param not in param_bounds:
        raise ValueError(f"Parameter {param} not found in param_bounds")
    
    lo, hi = param_bounds[param]
    initials = np.linspace(lo, hi, n_initials)
    param_dir = os.path.join(sweep_output_root, param)
    os.makedirs(param_dir, exist_ok=True)
    agg_rows = []
    run_colors = plt.cm.inferno(np.linspace(0.15, 0.95, len(initials)))
    
    for i, init in enumerate(initials):
        opt_params = {param: float(init)}
        fixed_run = fixed_params.copy()
        for parameter, value in best_fit_params.items():
            if parameter != param:
                fixed_run[parameter] = value
        
        log_file_run = os.path.join(param_dir, f"run_{i:03d}_optimisation_log.csv")
        trace_file_run = os.path.join(param_dir, f"run_{i:03d}_optimisation_trace.csv")
        print(f"Running sweep for {param}: init #{i} = {init}")
        
        try:
            best_opt_params_run, loss_history_run, param_errors_run = gradient_descent.fit_streamline(
                opt_params,
                fixed_run,
                data,
                uncertainties,
                distance,
                learning_rate=learning_rate,
                param_bounds={param: param_bounds[param]},
                n_epochs=n_epochs,
                info_every=10,
                loss_threshold=0.05,
                loss_threshold_epochs=5,
                gradient_tol=gradient_tol,
                gradient_tol_epochs=5,
                early_stopping_patience=80,
                log_file=log_file_run,
                trace_file=trace_file_run,
                trace_every=1,
                loss_method=loss_method,
                output_uncertainties=False,
            )
        except Exception as e:
            print(f"Run failed for {param} init {init}: {e}")
            continue
        
        if os.path.exists(log_file_run):
            df = pd.read_csv(log_file_run)
            df["param_name"] = param
            df["run_index"] = i
            df["initial_guess"] = float(init)
            agg_rows.append(df)
        else:
            print(f"Expected log file not found: {log_file_run}")
    
    # Aggregate and plot results
    if len(agg_rows) > 0:
        df_all = pd.concat(agg_rows, ignore_index=True)
        out_csv = os.path.join(param_dir, f"sweep_results_lr{learning_rate}.csv")
        df_all.to_csv(out_csv, index=False)
        
        fig, (ax_param) = plt.subplots(1, 1, figsize=(9, 4), sharex=True)
        for idx, group in df_all.groupby('run_index'):
            color = run_colors[int(idx) % len(run_colors)]
            label = f"init={group['initial_guess'].iloc[0]:.3g}"
            if param in group.columns:
                ax_param.plot(group['epoch'], group[param], color=color, alpha=0.9, lw=1.8, label=label)
        
        best_fit_value = best_fit_params[param]
        ax_param.axhline(best_fit_value, color='grey', linestyle='--', label=f'Best-fit {param}={best_fit_value:.3g}')
        
        ax_param.set_ylabel(param)
        ax_param.set_xlabel('Epoch')
        ax_param.set_title(f'Parameter sweep: {param}, learning_rate={learning_rate}')
        ax_param.grid(True, alpha=0.3)
        ax_param.legend(fontsize='small', ncol=2)
    
        
        save_png = os.path.join(param_dir, f'sweep_plot_lr{learning_rate}.png')
        fig.tight_layout()
        fig.savefig(save_png, dpi=200, bbox_inches='tight')
        plt.show()
        print(f"Saved {out_csv} and {save_png}")
        return df_all
    else:
        print(f"No successful runs for parameter {param}")
        return None

## Sweep through initial guesses for one parameter at a time
Leave all the rest of the parameters fixed at their best-fit value

In [ ]:
# r0
run_param_sweep('r0', n_initials=8, n_epochs=100, learning_rate=0.005)
run_param_sweep('r0', n_initials=8, n_epochs=100, learning_rate=0.01)
run_param_sweep('r0', n_initials=8, n_epochs=100, learning_rate=0.02)
run_param_sweep('r0', n_initials=8, n_epochs=100, learning_rate=0.04)

In [ ]:
# theta0
run_param_sweep('theta0', n_initials=8, n_epochs=100, learning_rate=0.005)
#run_param_sweep('theta0', n_initials=8, n_epochs=100, learning_rate=0.01)
#run_param_sweep('theta0', n_initials=8, n_epochs=100, learning_rate=0.02)
#run_param_sweep('theta0', n_initials=8, n_epochs=100, learning_rate=0.04)

In [ ]:
# phi0
#run_param_sweep('phi0', n_initials=8, n_epochs=100, learning_rate=0.005)
run_param_sweep('phi0', n_initials=8, n_epochs=100, learning_rate=0.01)
#run_param_sweep('phi0', n_initials=8, n_epochs=100, learning_rate=0.02)
#run_param_sweep('phi0', n_initials=8, n_epochs=100, learning_rate=0.04)

In [ ]:
# log_omega
#run_param_sweep('log_omega', n_initials=8, n_epochs=100, learning_rate=0.005)
run_param_sweep('log_omega', n_initials=8, n_epochs=100, learning_rate=0.01)
#run_param_sweep('log_omega', n_initials=8, n_epochs=100, learning_rate=0.02)
#run_param_sweep('log_omega', n_initials=8, n_epochs=100, learning_rate=0.04)

In [ ]:
# v_r0
run_param_sweep('v_r0', n_initials=8, n_epochs=100, learning_rate=0.005)
#run_param_sweep('v_r0', n_initials=8, n_epochs=200, learning_rate=0.01)
#run_param_sweep('v_r0', n_initials=8, n_epochs=100, learning_rate=0.02)
#run_param_sweep('v_r0', n_initials=8, n_epochs=100, learning_rate=0.04)